# Predicting iron deficiency from a blood count

## From Zero to Hero Session 1: local training on real healthcare data

In this session we're building a clinical machine learning pipeline on real patient data from the US National Health and Nutrition Examination Survey (NHANES). This is a real healthcare dataset which is freely downloadable. (So go experiment more with it for your own projects!)

### The clinical question

The **complete blood count (CBC)**, or full blood count (FBC) in British terminology, is the most common medical laboratory test in the world. It's very low-cost and is often the first "bloods and vitals" collected for any patient.

**Serum ferritin concentration** is the standard marker for iron stores, though is more expensive than the CBC. It is typically ordered when the CBC indicates possible iron deficiency.

> **OUR TASK:** can the blood count a patient *already has* tell us who needs a
> ferritin test they *have not yet had*? --> Screening objective

Screening questions carry a cost asymmetry: Missing an iron-deficient patient leads to anaemia and, in pregnancy, harms the fetus. On the other hand, ordering an unnecessary ferritin test only loses the health service a few pounds.
Thus, we'd like to prioritise a fairly *sensitive* test rather than a *specific* test.

### What we will do

1. Acquire and merge real survey/EHR-style data (Parts 1 and 2)
2. Look at an example of how a machine learning result can look deceivingly good (Part 3)
3. Local PyTorch implementation: `Dataset`, `DataLoader`, an MLP, a training loop (Parts 4 and 5)
4. Evaluation inspired by what's clinically impactful: calibration, operating points, decision curves (Parts 6 and 7)
5. Some basic fairness auditing with `fairlearn` (Part 8)

### Side-note
For a different task (Diabetes detection) on the same dataset, check out our Flower Hub app! https://flower.ai/apps/bloodcounts/nhanes-t2d-fedmap-fl

---
## Part 0: setup

You need `torch`, `scikit-learn`, `pandas`, `matplotlib` and `fairlearn`. Can run in Google Colab or you can download the notebook and run it on your laptop (no GPU needed).

(In the below we do set the PyTorch device to CUDA (NVIDIA GPU), MPS (Apple Silicon) or CPU (fallback). Though not really necessary for the proposed small network, maybe good practice to use anyway in case you'd like to scale up). MPS handles `float32` but not `float64`, and every tensor we build
is `float32`, so the same code runs on all three.)

In [ ]:
# Installing dependencies
# (can comment out line below if you're running in your local environment):
!pip install torch scikit-learn pandas matplotlib fairlearn

import urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

# reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()      # Apple Silicon
          else "cpu")

print(f"torch {torch.__version__} | device: {DEVICE}")

# some plotting style settings
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "legend.frameon": False,
})
BLUE, RED, GREY, GREEN = "#1f6feb", "#d1495b", "#8b949e", "#2a9d8f"

---
## Part 1: getting the data

The US CDC runs NHANES continuously. Every 2-year **cycle** examines a fresh
national probability sample with questionnaires, a physical examination and a
laboratory panel. The data are public, de-identified and need no registration,
which makes NHANES the closest thing to real EHR data you may easily download.

### File naming

Data ship as SAS transport files (`.XPT`), one per **component**, with a suffix
per cycle:

| Component | 2005-06 | 2015-16 | 2017-18 | 2021-23 |
|---|---|---|---|---|
| Demographics | `DEMO_D` | `DEMO_I` | `DEMO_J` | `DEMO_L` |
| Blood count | `CBC_D` | `CBC_I` | `CBC_J` | `CBC_L` |
| Ferritin | `FERTIN_D` | `FERTIN_I` | `FERTIN_J` | `FERTIN_L` |

Every record carries **`SEQN`**, the respondent ID, and that is the key we merge
on. Treat it as the patient identifier that lets you join a lab table to a
demographics table, the same join you would write against an EHR warehouse.

In [ ]:
CACHE = Path("nhanes_cache"); CACHE.mkdir(exist_ok=True)

# Four NHANES cycles
CYCLES = {"2005-2006": "D", "2015-2016": "I", "2017-2018": "J", "2021-2023": "L"}
CYCLE_DIR = {"D": "2005", "I": "2015", "J": "2017", "L": "2021"}
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{yr}/DataFiles/{fn}.xpt"


def fetch_xpt(name, cycle_letter):
    """Download (and cache) one NHANES component, returned as a DataFrame."""
    fn = f"{name}_{cycle_letter}"
    local = CACHE / f"{fn}.xpt"

    if not local.exists():
        print(f"  downloading {fn} ...", end=" ")
        urllib.request.urlretrieve(BASE.format(yr=CYCLE_DIR[cycle_letter], fn=fn), local)
        print(f"{local.stat().st_size / 1024:,.0f} KB")

    df = pd.read_sas(local, format="xport")
    df.columns = [c.upper() for c in df.columns]
    return df


print("Fetching NHANES components (cached after first run):")

raw = {}
for label, letter in CYCLES.items():
    print(f"{label}:")
    raw[label] = {"demo": fetch_xpt("DEMO", letter),
                  "cbc":  fetch_xpt("CBC", letter),
                  "fer":  fetch_xpt("FERTIN", letter)}

print("\nDone.")

In [ ]:
# What does a raw NHANES lab table look like?
cbc_example = raw["2017-2018"]["cbc"]
print(f"CBC_J: {cbc_example.shape[0]:,} rows x {cbc_example.shape[1]} columns")

cbc_example.head(3)

The column names are opaque codes such as `LBXHGB` and `LBXMCVSI`. An online
codebook documents every NHANES variable, and the naming follows a system
(`LBX` marks a laboratory result, `SI` marks SI units). Below is the translation
for the blood count, which is the whole feature space of this tutorial.

In [ ]:
CBC_LABELS = {
    "LBXWBCSI": "White blood cells (1000 cells/uL)",
    "LBXRBCSI": "Red blood cells (million cells/uL)",
    "LBXHGB":   "Haemoglobin (g/dL)",
    "LBXHCT":   "Haematocrit (%)",
    "LBXMCVSI": "MCV, mean corpuscular volume (fL)",
    "LBXMC":    "MCHC, mean corpusc. Hb concentration (g/dL)",
    "LBXMCHSI": "MCH, mean corpuscular haemoglobin (pg)",
    "LBXRDW":   "RDW, red cell distribution width (%)",
    "LBXPLTSI": "Platelet count (1000 cells/uL)",
    "LBXMPSI":  "Mean platelet volume (fL)",
    "LBDLYMNO": "Lymphocytes (1000 cells/uL)",
    "LBDNENO":  "Neutrophils (1000 cells/uL)",
    "LBDMONO":  "Monocytes (1000 cells/uL)",
}
CBC_VARS = list(CBC_LABELS)

pd.Series(CBC_LABELS, name="meaning").to_frame()

### Quick biology: CBC and iron deficiency

If you remember nothing else about the biology, remember these:

- **MCV** (mean corpuscular volume), the average red cell *size*. A cell needs
  iron to make haemoglobin, and without iron it comes out small ("microcytic"),
  so MCV falls.
- **MCH / MCHC**, haemoglobin *content* and *concentration* per cell. Iron
  deficiency leaves cells pale ("hypochromic") and both indices fall.
- **RDW** (red cell distribution width), the *variability* in cell size. Early
  iron deficiency produces a mixed population of normal and small cells, so RDW
  rises before MCV falls. It works as an early-warning index.
- **HGB** (haemoglobin concentration), if your cells are smaller you also have less haemoglobin in them. This is the main parameter that impacts the health of the patient.

So it goes: low iron -> can't make haemoglobin -> small/underfilled red cells -> can't carry as much oxygen around the body

A low MCV, MCH, or HGB picture is the classic fingerprint of iron deficiency. Our network has to learn that pattern from data.

---
## Part 2: cohort definition, and a trap in the survey design

Clinical projects are won or lost at the cohort definition.

### Sidenote on traps: ferritin is measured in a *subsample* only

NHANES does not run every assay on every participant. In most cycles the lab
measured ferritin only in children aged 1 to 5 and females aged 12 to 49. Pool
the cycles naively, compute "iron deficiency prevalence", and you compare
different populations across cycles without noticing.

Let's have a look at that:

In [ ]:
rows = []
for label in CYCLES:
    d, f = raw[label]["demo"], raw[label]["fer"]
    m = d[["SEQN", "RIAGENDR", "RIDAGEYR"]].merge(f[["SEQN", "LBXFER"]], on="SEQN")
    m = m.dropna(subset=["LBXFER"])

    rows.append({"cycle": label,
                 "n with ferritin": len(m),
                 "% female": round(100 * (m.RIAGENDR == 2).mean()),
                 "age min": int(m.RIDAGEYR.min()),
                 "age max": int(m.RIDAGEYR.max())})

pd.DataFrame(rows)

Three of the four cycles are 85 to 93% female and capped at age
49, which is the restricted subsample. **2017-2018 differs** in that the lab measured ferritin across all ages and both sexes.

A "prevalence trend over time" computed from these four cycles without reading
the documentation would report an artefact of *who got tested*, not a change in
population health. Public-data analyses go wrong this way a lot.
Similarly in an EHR database: the sickest patients get the most tests, so any lab value in an EHR is present *non-randomly*.

### Choosing the cohort

We restrict to **women aged 18 to 49**, for two reasons:

1. **Clinical.** Iron deficiency concentrates here, through menstrual blood loss
   and pregnancy. In the data below it affects about 1 in 6 women in this band,
   against about 1 in 90 men under 50.
2. **Consistency.** This is the one group present in *every* cycle, so our four
   "sites" stay comparable.

In [ ]:
# NHANES codes: RIAGENDR 1=male 2=female; RIDRETH1 = race/ethnicity (consistent
# we also code ethnicity for later
# across all four cycles, unlike RIDRETH3 which added a category later).
ETHNICITY = {1.0: "Mexican American", 2.0: "Other Hispanic", 3.0: "White",
             4.0: "Black", 5.0: "Other/Multi"}

frames = []
for label in CYCLES:
    d, c, f = raw[label]["demo"], raw[label]["cbc"], raw[label]["fer"]
    m = (d[["SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH1"]]
         .merge(c[["SEQN"] + CBC_VARS], on="SEQN")          # inner join on patient ID
         .merge(f[["SEQN", "LBXFER"]], on="SEQN"))

    m = m[(m.RIAGENDR == 2) & m.RIDAGEYR.between(18, 49)]   # cohort filter
    frames.append(m.assign(site=label))

merged = pd.concat(frames, ignore_index=True)
merged["ethnicity"] = merged.RIDRETH1.map(ETHNICITY)

print(f"After merge + cohort filter: {len(merged):,} women aged 18-49")

merged.groupby("site").size().to_frame("n")

### Missing data

Look at what is absent before modelling. This step is mandatory in EHR work and
usually alarming. NHANES is unusually complete because the CDC designed it as a
survey rather than accumulating it as a by-product of care.

In [ ]:
miss = (merged[CBC_VARS + ["LBXFER"]].isna().mean() * 100).round(2).to_frame("% missing")
miss["label"] = [CBC_LABELS.get(i, "Ferritin (ng/mL)  <- our label source") for i in miss.index]

miss

In [ ]:
# Rows with no ferritin have no label, so they cannot train or test anything.
labelled = merged.dropna(subset=["LBXFER", "ethnicity"])

# How is the CBC missingness distributed? Count gaps per patient.
gaps = labelled[CBC_VARS].isna().sum(axis=1)

print(f"Labelled rows: {len(labelled):,}")
print(f"Complete blood count: {(gaps == 0).sum():,} ({100 * (gaps == 0).mean():.2f}%)")

gaps.value_counts().sort_index().rename_axis("missing CBC values").to_frame("patients")

So in this dataset we are lucky in that we find:

**The gaps are whole-panel, not scattered.** No patient is missing one or two CBC indices. A patient has all 13 values, or is missing 3 (the differential white count, reported from a second channel), or is missing 12 or 13. The analyser processed the sample or it did not. Nothing here looks like a value that was measured and then lost.

**The loss is 0.5%.** Dropping every incomplete CBC costs us about 30
patients out of 5,750. Not so bad!

So in this case we use complete cases and skip imputation (i.e. filling missing values with educated guesses).

**When you should impute instead:** Push the missing fraction to say 10 or 20% and dropping rows starts to cost power and to bias the cohort, because the patients with missing labs differ from those without. At that point use a principled method (iterative/MICE imputation fitted on the training split alone) and report the model with and without it. Add a missingness indicator column too, since in an EHR the fact that a test is absent often carries *as much signal* as its value. E.g. some extended CBC results like reticulocyte counts (young red cells) are almost always missing unless specially ordered *because* there was a suspicion of something going on.

For further reading on imputation and its subtleties, I recommend this paper by workshop co-organiser Mike Roberts: https://www.nature.com/articles/s43856-023-00356-z

In [ ]:
before = len(merged)
cohort = labelled.dropna(subset=CBC_VARS).reset_index(drop=True)

print(f"Dropped {before - len(cohort):,} rows lacking a label, an ethnicity code "
      f"or a complete blood count.")
print(f"Analysis cohort: {len(cohort):,} patients.")

### One site is "our hospital"

Session 2 will distribute this task across all four sites. For **this** session
we act as a single institution, so we now put three of the four cycles away and
leave them alone.

We take **2017-2018** as our site.

Think of it as you hold your own patients, and other hospitals' data sit behind a governance boundary you cannot cross.

In [ ]:
HOME_SITE = "2017-2018"

home = cohort[cohort.site == HOME_SITE].reset_index(drop=True).copy()
away = cohort[cohort.site != HOME_SITE].reset_index(drop=True).copy()

print(f"Our site ({HOME_SITE}): {len(home):,} patients")
print(f"Other sites (closed to us for now): {len(away):,} patients across "
      f"{away.site.nunique()} institutions")

pd.DataFrame({
    "n": home.ethnicity.value_counts(),
    "%": (home.ethnicity.value_counts(normalize=True) * 100).round(1),
})

---
# Part 3: Trying an easier task first

## 3.1 Iron defiency leads to anaemia, so that signal should be stronger

Iron deficiency develops slowly towards anaemia. So in a sense, anaemia is *severe* iron deficiency and should carry stronger signal than the more subtle early iron deficiency. Let's look at that first, why don't we!

The World Health Organization defines **anaemia** by a haemoglobin threshold, and for non-pregnant women that threshold is `Hb < 12 g/dL`.

The condition is well defined, clinically meaningful and common. Let's try to predict it from the CBC. As a first test we're going to use a Logistic Regression model:

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve

# One helper for every logistic-regression comparator in this notebook.
lr_pipe = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

anaemia_df = home.assign(anaemia=(home.LBXHGB < 12.0).astype(int)) # label

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split( # quick split into training and testing data
    anaemia_df[CBC_VARS], anaemia_df.anaemia,
    test_size=0.30, stratify=anaemia_df.anaemia, random_state=SEED)

print(f"Anaemia prevalence at our site: {100 * anaemia_df.anaemia.mean():.1f}% "
      f"({anaemia_df.anaemia.sum():,} of {len(anaemia_df):,})")

In [ ]:
pipe = lr_pipe().fit(Xa_tr, ya_tr)
p_anaemia = pipe.predict_proba(Xa_te)[:, 1]
auc_anaemia = roc_auc_score(ya_te, p_anaemia)

print(f"Test AUROC for anaemia: {auc_anaemia:.4f}")

In [ ]:
fpr, tpr, _ = roc_curve(ya_te, p_anaemia)

fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot(fpr, tpr, color=BLUE, lw=2.4, label=f"Our model (AUROC = {auc_anaemia:.4f})")
ax.plot([0, 1], [0, 1], color=GREY, ls=":", lw=1.2, label="Chance")

ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate (sensitivity)")
ax.set_title("Anaemia prediction from a blood count")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 3.2 Wow that's VERY good. Hmm ....

An AUROC around 1.0 on real patient data.

In machine learning, when a results looks near-perfect, it's almost always wrong.
General good practice is to check the following, in descending order of likelihood:

1. **Label leakage**, where a feature encodes the answer.
2. **A split error**, where the same patients sit in train and test.
3. A task so easy that nobody needed a model for it.

Thankfully logistic regression is very simple and we can directly read off what the model is using for its predictions:

In [ ]:
coefs = pd.Series(pipe[-1].coef_[0], index=CBC_VARS).sort_values(key=np.abs, ascending=False)

pd.DataFrame({"coefficient": coefs.round(2),
              "description": [CBC_LABELS[i] for i in coefs.index]})

Seems like it's primarily using Haemoglobin...

Now re-read our label definition:

```python
anaemia_df = home.assign(anaemia=(home.LBXHGB < 12.0).astype(int))
```

We asked the model to predict a **threshold on haemoglobin**, and we handed it
**haemoglobin** as a feature. The model learned no medicine. It learned to
compare a number to 12, and we could have written that in one line without
PyTorch, scikit-learn or a workshop.

This is **label leakage**, the most common serious bug in clinical ML.

The bug is obvious here because we wrote the label ourselves one cell earlier. In real projects it often hides if you don't check carefully.
Some examples:
- Predicting sepsis, with the antibiotic order in the feature set. The clinician
  gave the drug *because* they already suspected sepsis.
- Predicting a diagnosis from billing codes entered *after* the diagnosis.
- Predicting mortality using "discharge destination = mortuary". (yikes ...)
- Predicting deterioration from a lab that gets ordered only *when* the patient
  deteriorates, so the missingness leaks the label. (the reticulocyte count mentioned above in the missingness section)

**The reflex to build:** for every feature, ask *"would this value exist, with
this value, at the moment I need the prediction?"*

**It is imperative that ML researchers, bioinformaticians, and clinicians work together in these types of projects to make sure realistic and good practice is followed throughout.**

---
# Part 4: Iron defifiency detection (the main task)

Ok so we saw above that anaemia is actually directly encoded within a CBC. But what if your patient presents with fatigue and trouble concentrating but their HGB, MCV, and MCH don't trigger below the reference values? She might still be iron-deficient but it's undetected in the classic CBC interpretation.

We will now try to find if we can apply better models on the CBC to detect even this more subtle iron defifiency which may not *yet* have led to anaemia.
Our label is **serum ferritin concentration < 15 µg/L**, which is the WHO threshold for depleted iron stores in adults.

Unlike the anaemia label, ferritin is a separate assay and thus sits outside the input feature set of the model.

**The task thus is:**
> Given the standard CBC parameters + age, detect if the patient currently has a ferritin concentration < 15 µg/L.

In [ ]:
FEATURES = CBC_VARS + ["RIDAGEYR"]          # 13 blood-count indices + age
FEATURE_LABELS = {**CBC_LABELS, "RIDAGEYR": "Age (years)"}

data = home.assign(iron_deficient=(home.LBXFER < 15.0).astype(int))
n_pos = int(data.iron_deficient.sum())

print(f"Site              : {HOME_SITE}")
print(f"Patients          : {len(data):,}")
print(f"Iron deficient    : {n_pos:,} ({100 * data.iron_deficient.mean():.1f}%)")
print(f"Iron replete      : {len(data) - n_pos:,}")
print(f"Class ratio       : 1 positive per {(len(data) - n_pos) / n_pos:.1f} negatives")

We see that our classes are imbalanced, i.e. only 17.3% of patients are actually iron defifiency. This is important to check, because a model that answers "not deficient" for everybody would still score **83% accuracy**.

## 4.2 Look at the data before training the model

Does the textbook fingerprint, small pale variable red cells, show up?

In [ ]:
show = ["LBXHGB", "LBXMCVSI", "LBXRDW", "LBXMCHSI", "LBXFER", "LBXPLTSI"]
titles = ["Haemoglobin (g/dL)", "MCV (fL)", "RDW (%)", "MCH (pg)",
          "Ferritin (µg/L)", "Platelets (1000/uL)"]

fig, axes = plt.subplots(2, 3, figsize=(11.5, 6.2))

for ax, var, title in zip(axes.ravel(), show, titles):
    dep = data.loc[data.iron_deficient == 1, var]
    rep = data.loc[data.iron_deficient == 0, var]

    if var == "LBXFER":
        bins = np.logspace(0, np.log10(400), 45)
        ax.set_xscale("log")
    else:
        bins = np.linspace(*np.percentile(data[var], [0.5, 99.5]), 40)

    ax.hist(rep, bins=bins, color=GREY, alpha=0.65, density=True, label="Iron replete")
    ax.hist(dep, bins=bins, color=RED, alpha=0.65, density=True, label="Iron deficient")
    ax.set_title(title, fontsize=10)
    ax.set_yticks([])

    if var in ("LBXHGB", "LBXMCHSI"):
        ax.set_ylabel("Density\n(each group scaled\nto its own total)", fontsize=8)
    if var == "LBXFER":
        ax.axvline(15, color="k", ls="--", lw=1.3)
        ax.annotate("label\nthreshold", (15, ax.get_ylim()[1] * 0.55), xytext=(6, 0),
                    textcoords="offset points", fontsize=7.5)

axes[0, 0].legend(fontsize=8.5, loc="upper left")
fig.suptitle("Iron deficiency separates on red-cell indices",
             fontsize=11.5, y=0.99)
fig.tight_layout()
plt.show()

The biology shows:

- **Haemoglobin, MCV and MCH**: the deficient distribution shifts left, so
  anaemic, microcytic and hypochromic.
- **RDW** shifts right, so cell size varies more, as expected.
- **Ferritin** shows the label threshold. Note the log scale, because ferritin
  skews hard to the right.

The distributions **overlap substantially**, which is why this is a real
prediction problem with a ceiling well below perfect, and again why the leaky anaemia
model's near-perfect curve should have looked implausible.

## 4.3 Baseline

Establish what different easier input feature combinations achieve first:

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

quick_auc = lambda cols: cross_val_score(lr_pipe(), data[cols], data.iron_deficient,
                                         cv=cv, scoring="roc_auc").mean()

baselines = {
    "Age only":                          quick_auc(["RIDAGEYR"]),
    "Haemoglobin only":                  quick_auc(["LBXHGB"]),
    "3 red-cell indices (Hb, MCV, RDW)": quick_auc(["LBXHGB", "LBXMCVSI", "LBXRDW"]),
    "Full blood count + age":            quick_auc(FEATURES),
}

pd.Series(baselines, name="5-fold CV AUROC").round(3).to_frame()

- **Age alone lands near 0.5**, so random.
- **Haemoglobin alone** already carries strong signal, around 0.87, because iron
  deficiency causes anaemia.
- **The full count** adds a modest but consistent margin over Hb alone, roughly
  +0.02, and most of that arrives with MCV and RDW.

## 4.4 Splitting the data into training-validation-test, by patient and stratified

Two rules:

1. **Split by patient, never by row.** We hold one row per patient, so this is
   easy here. In longitudinal EHR data every record from a patient must land in
   the same fold, or the model could memorise individuals.
2. **Stratify on the outcome.** Not as important but can be useful: At 17% positives, a random split could leave the test set with a noticeably different prevalence and make metrics unstable.

We use a three-way split: **train** to fit weights, **validation** for early
stopping and threshold choice, **test** for evaluation and metrics reporting.

In [ ]:
train_df, test_df = train_test_split(
    data, test_size=0.20, stratify=data.iron_deficient, random_state=SEED)
train_df, val_df = train_test_split(
    train_df, test_size=0.20, stratify=train_df.iron_deficient, random_state=SEED)

splits = {"train": train_df, "validation": val_df, "test": test_df}

pd.DataFrame({
    "n": [len(d) for d in splits.values()],
    "positives": [int(d.iron_deficient.sum()) for d in splits.values()],
    "prevalence %": [round(100 * d.iron_deficient.mean(), 1) for d in splits.values()],
}, index=list(splits))

## 4.5 Scaling: fit on training data only

Another classic leakage mistake:
Compute the mean and standard deviation for scaling features on the **whole dataset** and
information from your test patients enters the training pipeline. Your test score
turns optimistic and you cannot tell by how much.

Hence, the transform must be learned on train and merely *applied* to validation
and test. Dropping incomplete blood counts in Part 2 needs no such care, because
that decision looks at one row at a time and fits nothing. Every transform that
*fits* a parameter, whether a scaler, an imputer, a feature selector or a
discretiser, belongs inside the training split.

In [ ]:
mu = train_df[FEATURES].mean()
sigma = train_df[FEATURES].std().replace(0, 1.0)

def prepare(df):
    """Standardise with TRAIN mean/sd. No imputation: the cohort is complete."""
    X = ((df[FEATURES] - mu) / sigma).values.astype(np.float32)
    return X, df.iron_deficient.values.astype(np.float32)

X_train, y_train = prepare(train_df)
X_val,   y_val   = prepare(val_df)
X_test,  y_test  = prepare(test_df)

assert not np.isnan(X_train).any(), "complete-case cohort should contain no NaN"

print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")
print(f"\nTrain after scaling: mean {X_train.mean():+.3f}, sd {X_train.std():.3f}")
print(f"Test  after scaling: mean {X_test.mean():+.3f}, sd {X_test.std():.3f}")

## 4.6 PyTorch `Dataset` and `DataLoader`

A `Dataset` answers two questions: how many samples do you have (`__len__`) and
give me sample *i* (`__getitem__`). A `DataLoader` wraps it and handles batching,
shuffling and, if you ask, parallel loading.

Our data fit in memory, so we write the explicit `Dataset` class once, because
that is the pattern you need for data that does *not* fit, and because each
Flower client in Session 2 wraps its own `DataLoader` exactly like this.

In [ ]:
class TabularDataset(Dataset):
    """Minimal tabular dataset: features -> float32 vector, label -> float32 scalar."""

    def __init__(self, X, y):
        self.X, self.y = torch.from_numpy(X), torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64

# Shuffle the training loader so batches differ each epoch.
train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE,
                          shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(TabularDataset(X_val, y_val), batch_size=256)
test_loader = DataLoader(TabularDataset(X_test, y_test), batch_size=256)

xb, yb = next(iter(train_loader))
print(f"One batch: features {tuple(xb.shape)}, labels {tuple(yb.shape)}")
print(f"Batches per epoch: {len(train_loader)}")

## 4.7 The model

A **multi-layer perceptron** with two hidden layers, ReLU activations and dropout
for regularisation. Roughly 3,000 parameters for about 800 training patients,
which is tiny by deep learning standards.

In [ ]:
class IronMLP(nn.Module):
    """Two-hidden-layer MLP for binary classification on tabular data."""

    def __init__(self, n_features, h1=64, h2=32, p_drop=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, h1), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(h1, h2),         nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(h2, 1),                       # single logit
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)              # (batch,) not (batch, 1)


model = IronMLP(len(FEATURES)).to(DEVICE)

print(model)
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters()):,} "
      f"for {len(train_df):,} training patients")

---
# Part 5: training

## 5.1 Loss, optimiser, and the class-imbalance decision

At 17% positives we face a choice. `BCEWithLogitsLoss` accepts a `pos_weight`
argument that up-weights positive cases and pushes the model to predict them more
readily.

In this example we don't use `pos_weight` and we will see later that that's fine, but feel free to try both.

Below leaves `pos_weight` commented out so you can experiment.

In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4          # L2 regularisation, important with small n
MAX_EPOCHS    = 200
PATIENCE      = 20            # early-stopping patience, in epochs

# pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()])
# criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion = nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE,
                             weight_decay=WEIGHT_DECAY)

print(f"Loss     : {criterion.__class__.__name__} (no class re-weighting)")
print(f"Optimiser: Adam, lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}")

## 5.2 The training loop

Written out explicitly rather than hidden in a framework like PyTorch Lightning, because Session 2 will
replace *this loop* with a federated one, and you should be able to see which
lines move to the client and which stay on the server.

In [ ]:
@torch.no_grad()
def evaluate_loader(model, loader):
    """Return (mean loss, probabilities, labels) for a data loader."""
    model.eval()
    losses, probs, labels = [], [], []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        losses.append(criterion(logits, yb).item() * len(yb))
        probs.append(torch.sigmoid(logits).cpu().numpy())
        labels.append(yb.cpu().numpy())

    labels = np.concatenate(labels)
    return sum(losses) / len(labels), np.concatenate(probs), labels


history = []
best_auroc, best_state, best_epoch, stale = -np.inf, None, 0, 0

for epoch in range(1, MAX_EPOCHS + 1):

    # ---- train one epoch ----
    model.train()                                   # dropout ON
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimiser.zero_grad()                       # clear old gradients
        loss = criterion(model(xb), yb)             # forward
        loss.backward()                             # backward
        optimiser.step()                            # update weights
        running += loss.item() * len(yb)

    # ---- validate ----
    val_loss, val_probs, val_labels = evaluate_loader(model, val_loader)
    val_auroc = roc_auc_score(val_labels, val_probs)

    history.append({"epoch": epoch, "train_loss": running / len(y_train),
                    "val_loss": val_loss, "val_auroc": val_auroc})

    # ---- early stopping on validation AUROC ----
    if val_auroc > best_auroc:
        best_auroc, best_epoch, stale = val_auroc, epoch, 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f"Early stop at epoch {epoch} (no gain for {PATIENCE} epochs).")
            break

    if epoch % 20 == 0 or epoch == 1:
        print(f"epoch {epoch:3d} | train {running / len(y_train):.4f} | "
              f"val {val_loss:.4f} | val AUROC {val_auroc:.4f}")

model.load_state_dict(best_state)                   # restore best checkpoint
h = pd.DataFrame(history)

print(f"\nBest validation AUROC {best_auroc:.4f} at epoch {best_epoch}. Weights restored.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0))

ax1.plot(h.epoch, h.train_loss, color=BLUE, lw=1.9, label="Train")
ax1.plot(h.epoch, h.val_loss, color=RED, lw=1.9, label="Validation")
ax1.axvline(best_epoch, color=GREY, ls="--", lw=1.2)
ax1.annotate(f"best epoch ({best_epoch})", (best_epoch, ax1.get_ylim()[1] * 0.95),
             xytext=(6, -4), textcoords="offset points", fontsize=8, color="#444")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("BCE loss")
ax1.set_title("Loss curves"); ax1.legend()

ax2.plot(h.epoch, h.val_auroc, color=GREEN, lw=2.0)
ax2.axvline(best_epoch, color=GREY, ls="--", lw=1.2)
ax2.annotate(f"best {h.val_auroc.max():.3f}", (best_epoch, h.val_auroc.max()),
             xytext=(8, -10), textcoords="offset points", fontsize=8, color="#444")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Validation AUROC")
ax2.set_title("Validation ranking performance")

fig.tight_layout()
plt.show()

---
# Part 6: standard classifier model metrics

Every threshold and hyperparameter came from
training and validation data, now we open the test set for evaluation.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, average_precision_score,
                             precision_recall_curve, brier_score_loss, balanced_accuracy_score)

test_loss, test_probs, test_labels = evaluate_loader(model, test_loader)
val_loss_f, val_probs, val_labels = evaluate_loader(model, val_loader)
auroc_mlp = roc_auc_score(test_labels, test_probs)

print(f"Test loss  : {test_loss:.4f}")
print(f"Test AUROC : {auroc_mlp:.4f}   ({len(test_labels):,} held-out patients)")

## 6.1 Quick word on "accuracy"

At 17% prevalence, predicting "nobody is deficient" scores about 83% accuracy
while finding **zero** patients. We can see that this can be misleading when placing it next to our model with some other better metrics:

In [ ]:
pred_05 = (test_probs >= 0.5).astype(int)
never = np.zeros_like(test_labels, dtype=int)

pd.DataFrame({
    "Predicts nobody deficient": [
        accuracy_score(test_labels, never),
        balanced_accuracy_score(test_labels, never),
        precision_score(test_labels, never, zero_division=0),
        recall_score(test_labels, never),
        f1_score(test_labels, never)],
    "Our MLP (threshold 0.5)": [
        accuracy_score(test_labels, pred_05),
        balanced_accuracy_score(test_labels, pred_05),
        precision_score(test_labels, pred_05, zero_division=0),
        recall_score(test_labels, pred_05),
        f1_score(test_labels, pred_05)],
}, index=["Accuracy", "Balanced Accuracy", "Precision (PPV)", "Recall (sensitivity)", "F1"]).round(3)

Also, our model is currently using an output threshold of 0.5 to trigger a positive class prediction. Note the pattern at that threshold: **precision looks good, recall
looks mediocre.** The model identifies a minority of deficient patients while
being right about most of those it flags. A probability model on an imbalanced
problem behaves this way naturally, because pushing a prediction above 0.5
demands strong evidence, so borderline cases land on the negative side.

For a screening test that is the wrong trade. Missing a case is the expensive
error, and **0.5 is a software default, not a clinical decision.** Nothing about
our problem says the cut belongs there. We will try to choose it deliberately in Part 7.

## 6.2 Threshold-free performance: ROC and precision-recall

It's usually good to plot the Receiver-Operating Characteristic (ROC) *and* the Precision-Recall (PR) curves, esepcially for an imbalanced task like this. The precision-recall curve reports: of the patients I flag, how many turn out deficient?


Sidenote: it's technically not advised (though every paper does it) to plot the ROC and PR curves and their metrics on the test set, as you will not be able to check all thresholds against a ground truth label in live deployment of a model. More on this: https://www.nature.com/articles/s42256-024-00817-7

See also the corresponding Github repo https://github.com/alonhzn/testAUC for a nice Python tool to check for ROC curve drift and stability.

In [ ]:
fpr_m, tpr_m, _ = roc_curve(test_labels, test_probs)
prec, rec, _ = precision_recall_curve(test_labels, test_probs)
ap = average_precision_score(test_labels, test_probs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))

ax1.plot(fpr_m, tpr_m, color=BLUE, lw=2.4, label=f"MLP (AUROC {auroc_mlp:.3f})")
ax1.plot([0, 1], [0, 1], color=GREY, ls=":", lw=1.2, label="Chance")
ax1.set_xlabel("False positive rate"); ax1.set_ylabel("Sensitivity")
ax1.set_title("ROC curve"); ax1.legend(loc="lower right", fontsize=8.5)

ax2.plot(rec, prec, color=BLUE, lw=2.4, label=f"MLP (AP {ap:.3f})")
ax2.axhline(test_labels.mean(), color=GREY, ls=":", lw=1.4,
            label=f"Prevalence ({test_labels.mean():.2f})")
ax2.set_xlabel("Recall (sensitivity)"); ax2.set_ylabel("Precision (PPV)")
ax2.set_title("Precision-recall curve"); ax2.legend(loc="upper right", fontsize=8.5)
ax2.set_ylim(0, 1.02)

fig.tight_layout()
plt.show()

## 6.3 Calibration, the metric clinical reviewers ask for and ML papers omit

AUROC measures *ranking* only. Halve every predicted probability and it does not
move. A clinician reading "35% risk" needs that to mean 35 out of 100 such
patients are deficient. That property is called **calibration**:

In [ ]:
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(test_labels, test_probs, n_bins=8, strategy="quantile")
brier = brier_score_loss(test_labels, test_probs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3),
                               gridspec_kw={"width_ratios": [1, 1.15]})

ax1.plot([0, 1], [0, 1], color=GREY, ls=":", lw=1.4, label="Perfect calibration")
ax1.plot(mean_pred, frac_pos, "o-", color=BLUE, lw=2.0, ms=6, label=f"MLP (Brier {brier:.3f})")
ax1.set_xlabel("Mean predicted probability")
ax1.set_ylabel("Observed fraction deficient")
ax1.set_title("Calibration (reliability diagram)")
ax1.legend(loc="upper left", fontsize=8.5)

ax2.hist(test_probs[test_labels == 0], bins=30, color=GREY, alpha=0.7, label="Iron replete")
ax2.hist(test_probs[test_labels == 1], bins=30, color=RED, alpha=0.7, label="Iron deficient")
ax2.set_xlabel("Predicted probability of iron deficiency")
ax2.set_ylabel("Patients")
ax2.set_title("Where the model puts its predictions")
ax2.legend(fontsize=8.5)

fig.tight_layout()
plt.show()

print(f"Brier score: {brier:.4f}  (lower is better; always-predict-prevalence "
      f"scores {test_labels.mean() * (1 - test_labels.mean()):.4f})")

Points on the diagonal mean you can believe the predicted probabilities. Points
**below** the diagonal mean over-confident predictions, where predicted risk
exceeds the observed rate. Points **above** mean under-confident.

---
# Part 7: metrics that reflect clinical impact

Everything so far would satisfy an ML reviewer. However, what we care about is deployment in clinical practice, and thus: **if I switch this on next Monday, what happens to my patients?**

## 7.1 Choose the operating point from the clinical consequence

Our two errors are not equivalent:

| Error | Clinical consequence | Cost |
|---|---|---|
| **False negative**, miss a deficient woman | Untreated deficiency: fatigue, impaired cognition, poor pregnancy outcomes. Diagnosis delayed by months or years. | High |
| **False positive**, flag a replete woman | One unnecessary ferritin test (about 5 to 10 pounds) and brief reassurance. | Low |

The asymmetry is large for a screening triage tool, so we select the threshold
that achieves a **clinically specified minimum sensitivity**. For this example let's go with 90%.

Again, we choose the threshold on the validation set, not on test. Choosing it on
test would be a subtle version of the leakage we diagnosed in Part 3.

In [ ]:
TARGET_SENSITIVITY = 0.90

def threshold_for_sensitivity(y_true, probs, target):
    """Highest threshold that still reaches the target sensitivity (fewest false positives)."""
    ok = [t for t in np.unique(probs)
          if recall_score(y_true, (probs >= t).astype(int)) >= target]
    return max(ok) if ok else probs.min()


THRESHOLD = threshold_for_sensitivity(val_labels, val_probs, TARGET_SENSITIVITY)
pred_clin = (test_probs >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(test_labels, pred_clin).ravel()

print(f"Chosen on the VALIDATION set: threshold = {THRESHOLD:.4f}")
print(f"\nTest-set performance at this operating point ({len(test_labels)} patients):")
print(f"  Sensitivity (recall) : {tp / (tp + fn):.3f}   <- we targeted {TARGET_SENSITIVITY} on val")
print(f"  Specificity          : {tn / (tn + fp):.3f}")
print(f"  PPV (precision)      : {tp / (tp + fp):.3f}")
print(f"  NPV                  : {tn / (tn + fn):.3f}")

sens, spec = tp / (tp + fn), tn / (tn + fp)

In [ ]:
cm = np.array([[tn, fp], [fn, tp]])
labels = [["True negative\n(correctly not tested)", "False positive\n(unnecessary ferritin)"],
          ["False negative\n(MISSED deficiency)", "True positive\n(correctly identified)"]]

fig, ax = plt.subplots(figsize=(5.4, 4.4))
ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())

for i in range(2):
    for j in range(2):
        dark = cm[i, j] > cm.max() * 0.55
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=17,
                color="white" if dark else "#111", fontweight="bold")
        ax.text(j, i + 0.30, labels[i][j], ha="center", va="center", fontsize=7.8,
                color="white" if dark else "#333")

ax.set_xticks([0, 1], ["Not flagged", "Flagged"])
ax.set_yticks([0, 1], ["Iron replete", "Iron deficient"])
ax.set_xlabel("Model decision"); ax.set_ylabel("Truth (ferritin)")
ax.set_title(f"Confusion matrix at {TARGET_SENSITIVITY:.0%} target sensitivity")
ax.grid(False)
fig.tight_layout()
plt.show()

## 7.2 Translate the matrix into clinical workload

In [ ]:
n_test, n_true = len(test_labels), int(test_labels.sum())

impact = pd.DataFrame([
    {"Strategy": "Ferritin for every woman", "Ferritin tests": n_test,
     "Cases found": n_true, "Cases missed": 0, "Tests per case found": n_test / n_true},
    {"Strategy": "Ferritin for nobody", "Ferritin tests": 0,
     "Cases found": 0, "Cases missed": n_true, "Tests per case found": np.nan},
    {"Strategy": f"Ferritin only if model flags (sens {sens:.0%})",
     "Ferritin tests": int(tp + fp), "Cases found": int(tp), "Cases missed": int(fn),
     "Tests per case found": (tp + fp) / tp},
])
impact["Tests avoided vs test-all"] = n_test - impact["Ferritin tests"]
impact["% tests avoided"] = (100 * impact["Tests avoided vs test-all"] / n_test).round(1)

scale = 1000 / n_test
print("Per 1,000 women screened with the model:")
print(f"  {(tp + fp) * scale:5.0f} ferritin tests ordered   (vs 1,000 if we test everyone)")
print(f"  {tp * scale:5.0f} deficient women identified")
print(f"  {fn * scale:5.0f} deficient women MISSED")
print(f"  {fp * scale:5.0f} unnecessary tests")

impact.round(2)

So we avoid a substantial share of ferritin tests and
still find about 90% of cases, though **we do miss some.** In a real deployment those
missed women are the ethical centre of the discussion and often need to be further investigated to gauge model fairness and robustness. A screening tool is a
policy choice as much as a model.

## 7.3 The sensitivity and workload frontier

Other idea: rather than defending one threshold, show the whole trade-off and let the clinical
team choose:

In [ ]:
frontier = []
for t in np.arange(0.50, 1.001, 0.02):
    thr_t = threshold_for_sensitivity(val_labels, val_probs, t)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(
        test_labels, (test_probs >= thr_t).astype(int)).ravel()

    frontier.append({"target": t, "sens": tp_t / (tp_t + fn_t),
                     "tests_per_1000": 1000 * (tp_t + fp_t) / n_test,
                     "missed_per_1000": 1000 * fn_t / n_test,
                     "ppv": tp_t / (tp_t + fp_t) if (tp_t + fp_t) else np.nan})

fr = pd.DataFrame(frontier)
sel = fr.iloc[(fr.sens - sens).abs().argmin()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.3))

ax1.plot(fr.sens, fr.tests_per_1000, color=BLUE, lw=2.3)
ax1.axhline(1000, color=GREY, ls=":", lw=1.4)
ax1.annotate("test everyone (1,000)", (0.52, 1000), xytext=(0, -13),
             textcoords="offset points", fontsize=8, color="#444")
ax1.plot(sel.sens, sel.tests_per_1000, "o", color=RED, ms=9, zorder=5)
ax1.annotate(f"our choice\n{sel.sens:.0%} sens, {sel.tests_per_1000:.0f} tests",
             (sel.sens, sel.tests_per_1000), xytext=(-72, 14),
             textcoords="offset points", fontsize=8.2, color=RED)
ax1.set_xlabel("Sensitivity achieved"); ax1.set_ylabel("Ferritin tests per 1,000 women")
ax1.set_title("Workload rises steeply for the last few percent")

ax2.plot(fr.tests_per_1000, fr.missed_per_1000, color=RED, lw=2.3)
ax2.plot(sel.tests_per_1000, sel.missed_per_1000, "o", color=RED, ms=9, zorder=5)
ax2.set_xlabel("Ferritin tests per 1,000 women")
ax2.set_ylabel("Deficient women missed per 1,000")
ax2.set_title("The trade-off")

fig.tight_layout()
plt.show()

## 7.4 Decision curve analysis

Decision curve analysis (DCA) asks the question a health economist asks: across the
range of risk thresholds a clinician might plausibly use, does acting on this
model produce more net benefit than "test everyone" or "test nobody"?

Net benefit counts true positives, then subtracts false positives weighted by the
odds of the threshold probability. That weight encodes how many unnecessary tests
you accept as the price of one extra case found. At a threshold probability of
0.10, for instance, you are declaring that finding one case justifies nine
unnecessary tests.

Sidenote: we are using DCA here for evaluation of a screening tool. More often it is applied to gauge harm of a clinical intervention/treatment, which is a slightly different interpretation. See also https://en.wikipedia.org/wiki/Decision_curve_analysis

In [ ]:
def net_benefit(y_true, probs, pt):
    """Net benefit of testing when predicted risk >= pt."""
    flagged = probs >= pt
    tp_ = np.sum(flagged & (y_true == 1))
    fp_ = np.sum(flagged & (y_true == 0))
    return (tp_ - fp_ * pt / (1 - pt)) / len(y_true)

pts = np.linspace(0.0, 0.60, 200)
nb_model = [net_benefit(test_labels, test_probs, pt) for pt in pts]
nb_all = [net_benefit(test_labels, np.ones_like(test_probs), pt) for pt in pts]

# For funsies: Check the "test everyone" curve against its closed form:
#   NB_all(pt) = prevalence - (1 - prevalence) * pt / (1 - pt)
prev = test_labels.mean()
closed_form = prev - (1 - prev) * pts / (1 - pts)
assert np.allclose(nb_all, closed_form), "test-everyone curve is wrong"

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.6))

ax.plot(pts, nb_model, color=BLUE, lw=2.5, label="MLP model")
ax.plot(pts, nb_all, color=GREY, lw=1.8, label="Test everyone")
ax.axhline(0, color="k", lw=1.2, label="Test nobody")
ax.axvline(THRESHOLD, color=RED, ls=":", lw=1.5)
ax.annotate("our operating point", (THRESHOLD, 0.20), xytext=(7, 0),
            textcoords="offset points", fontsize=8.2, color=RED)

# Net benefit below zero means "worse than doing nothing", so the detail there
# carries no decision value
ax.set_ylim(-0.10, 0.25)
ax.set_xlim(0, 0.60)
ax.set_xlabel("Threshold probability (clinician's risk tolerance)")
ax.set_ylabel("Net benefit")
ax.set_title("Decision curve analysis")
ax.legend(fontsize=8.8, loc="upper right")
fig.tight_layout()
plt.show()

### Reading the curves

The diagnostic value of DCA lies in three places:

- **The size of the gap**, which is the net benefit you gain over the best
  default at your threshold. Read it in the plot's own units: net benefit is
  measured in true positives per patient, so a gap of 0.05 at pt = 0.10 means five
  extra cases found per 100 women screened, at no cost in unnecessary tests
  relative to testing everyone.
- **Where the gap opens.** Below pt = 0.02 the model adds almost nothing, because
  a clinician who considers ferritin tests nearly free should just order them. The
  model earns its place from roughly pt = 0.05 upward, where the ferritin test starts to cost more.
- **Whether the model ever drops below a default**, which happens when
  calibration fails. A miscalibrated model can fall below "test nobody" at some
  thresholds.

A useful exercise: deliberately miscalibrate. Multiply `test_probs` by 3, clip to
1, re-run the two cells above, and watch the curve cross below zero.

## 7.5 Feature importance (i.e. explainability)

We use
permutation importance: shuffle one feature and measure how much test AUROC
degrades. Different feature importance mechanisms exist. You might want to look into [SHAP](https://shap.readthedocs.io/en/latest/) as well.

In [ ]:
rng = np.random.default_rng(SEED)

@torch.no_grad()
def auroc_with_shuffled(col_idx, repeats=12):
    model.eval()
    scores = []

    for _ in range(repeats):
        Xp = X_test.copy()
        Xp[:, col_idx] = rng.permutation(Xp[:, col_idx])
        p = torch.sigmoid(model(torch.from_numpy(Xp).to(DEVICE))).cpu().numpy()
        scores.append(roc_auc_score(test_labels, p))

    return np.mean(scores)


imp = (pd.DataFrame({"feature": FEATURES,
                     "drop_in_auroc": [auroc_mlp - auroc_with_shuffled(i)
                                       for i in range(len(FEATURES))]})
       .sort_values("drop_in_auroc", ascending=False))
imp["description"] = imp.feature.map(FEATURE_LABELS)

fig, ax = plt.subplots(figsize=(7.4, 4.6))
ax.barh(range(len(imp)), imp.drop_in_auroc,
        color=[BLUE if v > 0 else GREY for v in imp.drop_in_auroc])
ax.set_yticks(range(len(imp)),
              [FEATURE_LABELS[f].split(",")[0].split(" (")[0] for f in imp.feature],
              fontsize=8.5)
ax.invert_yaxis()
ax.axvline(0, color="k", lw=1.0)
ax.set_xlabel("Drop in test AUROC when feature is shuffled")
ax.set_title("Permutation importance: does the model use the right physiology?")
fig.tight_layout()
plt.show()

imp.round(4).reset_index(drop=True)

The top of the ranking holds **haemoglobin, haematocrit, RDW and MCHC**, the
red-cell indices, which is the haematology of iron deficiency.

### An important caveat about this plot

You may notice **MCV ranks low**, near the bottom, despite the Part 4.2
histograms showing it separates the classes clearly. That is no contradiction. It
is a known limitation of permutation importance:

> When features correlate strongly, permutation importance splits the credit among
> them and can make each look individually unimportant.

MCV, MCH, MCHC and haematocrit are near-algebraic functions of one another.
Shuffling MCV alone barely hurts, because the model recovers the same information
from its correlated neighbours. **Low permutation importance means "this feature
is redundant given the others", not "this feature is uninformative".**

The practical lesson: use this plot to check that the model attends to the right
*physiological system*, not to rank individual features for clinical
interpretation. For the latter you need methods that handle correlation
explicitly, such as grouped permutation (shuffle all red-cell indices together),
conditional importance, or [SHAP](https://shap.readthedocs.io/en/latest/) with a correlation-aware background.

---
# Part 8: Brief look at fairness auditing

A model can post an excellent overall AUROC while working well for one group and
badly for another. Aggregate metrics hide that by construction. Regulators
increasingly require subgroup reporting, and it belongs in any clinical paper.

We audit across **race/ethnicity**, using [Fairlearn](https://fairlearn.org/).

In [ ]:
from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference

groups = test_df.ethnicity.values

mf = MetricFrame(
    metrics={"n": lambda y, p: len(y),
             "prevalence": lambda y, p: y.mean(),
             "AUROC": lambda y, p: roc_auc_score(y, p) if 0 < y.mean() < 1 else np.nan},
    y_true=test_labels, y_pred=test_probs, sensitive_features=groups)

thr_frame = MetricFrame(
    metrics={"sensitivity": recall_score,
             "specificity": lambda y, p: recall_score(1 - y, 1 - p),
             "PPV": lambda y, p: precision_score(y, p, zero_division=0),
             "flagged rate": selection_rate},
    y_true=test_labels, y_pred=pred_clin, sensitive_features=groups)

audit = pd.concat([mf.by_group, thr_frame.by_group], axis=1)
audit["n"] = audit["n"].astype(int)

audit.round(3)

In [ ]:
plot_cols = ["sensitivity", "PPV", "flagged rate"]
ok = audit[audit.n >= 20].sort_values("sensitivity")     # tiny groups are noise
x = np.arange(len(ok))

fig, ax = plt.subplots(figsize=(8.6, 4.3))
for i, (col, colour) in enumerate(zip(plot_cols, [BLUE, GREEN, GREY])):
    ax.bar(x + (i - 1) * 0.26, ok[col], width=0.25, color=colour, label=col)

ax.axhline(sens, color=RED, ls="--", lw=1.4)
ax.annotate(f"overall sensitivity {sens:.2f}", (len(ok) - 0.55, sens), xytext=(0, 5),
            textcoords="offset points", fontsize=8, color=RED, ha="right")
ax.set_xticks(x, [f"{g}\n(n={int(n)})" for g, n in zip(ok.index, ok.n)], fontsize=8.5)
ax.set_ylabel("Metric value")
ax.set_title("Performance by race/ethnicity at the clinical operating point")
ax.legend(fontsize=8.5, ncol=3)
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

print(f"Demographic parity difference (max - min flagged rate): "
      f"{demographic_parity_difference(test_labels, pred_clin, sensitive_features=groups):.3f}")
print(f"Sensitivity range across groups with n>=20: "
      f"{ok.sensitivity.min():.2f} to {ok.sensitivity.max():.2f}")

### How to read this audit

Sensitivity varies across groups. Before calling that a fairness failure, note the
subgroup sizes: several groups hold **fewer than 60 test patients and fewer than
15 positives**. A sensitivity of 0.80 against 1.00 in a group with 10 positive
cases is two patients, well inside sampling noise. In a real audit it would be snensible here to report **confidence intervals
on subgroup metrics**.

Two findings that do survive that caveat:.

**Prevalence differs genuinely across groups.** That is a real epidemiological
observation from the survey, and it is not a model artefact.

**Flagged rate follows prevalence.** A single global threshold flags a larger
share of patients in higher-prevalence groups. Whether that is *fair* depends on
the definition you adopt:

- **Equal flagged rates** (demographic parity) would require different thresholds
  per group, and it would under-test the group with the greatest need.
- **Equal sensitivity** (equal opportunity) means every group gets the same chance
  of having their deficiency caught (this would normally be the target for a screening tool like this).
- **Equal PPV** (predictive parity) is provably incompatible with equal
  sensitivity whenever prevalence differs across groups.

### The main point of this very basic audit
We excluded ethnicity from the features and disparities
appeared anyway, i.e. **blindness is not fairness**. It's generally a good idea to keep this in mind and measure fairness for your models. In a practical scenario this is normally compared to some alternative baseline, see also demographic parity difference and equalised odds difference in Fairlearn: https://fairlearn.org/v0.14/user_guide/assessment/common_fairness_metrics.html

---
# Main Takeaways from Session 1

### On the data science

1. **Investigate a suspiciously good result.** An AUROC of ~1.0 was a bug, not
   an achievement. Interrogate strong results at least as hard as weak ones.
2. **Ask of every feature: In my deployment scenario, would this exist, with this value, at prediction
   time?**
3. **Take care to fit transforms and parameters on training data, and hyperparameters and operating thresholds on validation data. Test is for evaluation only.**
4. **Accuracy is meaningless under class imbalance.** Predicting "nobody is iron deficient" scored
   83% here.
5. **Report calibration.**
6. **Choose the operating point from the clinical cost asymmetry, on the validation set.**

### On the clinical translation

- Metrics a clinician can act on: sensitivity at a stated operating point, tests
  per case found, cases missed per 1,000 screened.
- Missed cases are the ethical centre of a screening tool.